# smICA automation design and coverage

**Status:** implemented in v2.2.0. This notebook records the current command boundary, scientific-operation coverage, hidden state, protocol contracts and reproducibility policy. The TCP layer reuses existing GUI callbacks and does not introduce a parallel scientific calculation engine.

The transport uses loopback TCP, newline-delimited JSON, an explicit command registry and a thread-safe queue consumed on the GUI thread.


The initial design audit has been retained as implementation rationale, while the tables below describe the current v2.2.0 interface. Commands for extraction, statistical-filter setup, filter-library import, foreground-filter selection, Phot2conc calculation and export are operational. Complete publication workflows are currently orchestrated by scripts that call this public interface and maintain their own resume/report state.

## Audited surfaces

| Area | Files | Current implementation |
|---|---|---|
| Application entry point | `src/smICA/smICA_tool.py` | Parses launcher arguments, optionally starts the local automation service and processes commands in the Dear PyGui render loop. |
| PTU extraction | `src/smICA/Modes/Extract_from_PTU` | File loading, gates, statistical filtering, library state, batch extraction and export remain callback-backed. |
| Concentration | `src/smICA/Modes/Phot2conc` | Calibration, ROI construction, integrated/frame-wise calculation, display and exports share the existing mode state. |
| Common state | `src/smICA/Required` | Modes are dynamically mounted; the dispatcher serializes GUI work onto the owner thread. |
| Packaging | `setup.py`; installer scripts | Creates text-based Bash/BAT `smica-call` and `smica_call` scripts; launchers support GUI, `--test` and `--automation-server`. |
| User workflow | documentation notebooks | Manual and automated PTU-to-concentration paths use the same native batch callbacks. |

## Current execution boundary

The GUI remains the source of truth for workflow order and scientific execution. Public commands use stable scientific names rather than exposing arbitrary widget tags, but their handlers set the corresponding Dear PyGui values and invoke existing callbacks on the GUI thread. A user or workflow script defines the same operation sequence that would be performed interactively.

Validated commands bind to existing widgets, callbacks, mode objects and result attributes. Small adapters are used only when an existing callback needs deterministic arguments, normally relies on an interactive file dialog or does not expose a machine-readable result. Numerical code remains in its established execution path.

Scientific operations include PTU decoding, raster reconstruction, frame integration, decay gating, statistical-filter calculation and application, ROI construction, effective-volume calculation, photon-to-molecule conversion, concentration calculation, uncertainty calculation and numerical export. Pure presentation settings remain outside the workflow unless they change a scientific selection.

## Hidden state that must become explicit

- Input directory, selected file, selected channels and current mode.
- PTU record type, scanner identity, detector routing, geometry, pixel dwell time, marker count and integrated-frame count.
- Rows skipped, decay gates, statistical-filter enablement, background source/range/level, afterpulsing fraction, calculated weights and accepted filter state.
- Calibration values and uncertainties for lateral radius, axial ratio, effective volume and molecular brightness.
- ROI source, per-channel mask, automatic-ROI image source, smoothing, thresholds, cell/nucleus mode, selected connected component and channel-copy direction.
- Error mode, accumulated result rows, enabled exports, output directory and collision policy.

The corrected PicoHarp 300 T3 backend uses every reconstructed frame. Legacy PKL compatibility retains the historical effective-frame interpretation only when the explicit integrated-frame field is absent. Frame-range requests are valid only for a backend that exposes frame-resolved reconstruction; validation must reject unsupported legacy combinations. A complete workflow establishes this state by issuing the same validated command sequence from a fresh process.


## Automation command coverage

| Scientific operation | Public commands | Status |
|---|---|---|
| Mode and state inspection | `gui.select_mode`, `session.inspect`, `extract.get_state`, `phot2conc.get_state` | Implemented |
| PTU input and gate configuration | `extract.load_directory`, `extract.select_file`, `extract.set_gate`, `extract.set_rows_to_skip` | Implemented |
| Statistical-filter setup | `extract.set_statistical_filter`, `extract.open_filter_setup`, `extract.set_background_range`, `extract.calculate_filters`, `extract.accept_filters`, `extract.decline_filters` | Implemented |
| Decay library and foreground filter | `extract.open_decay_library`, `extract.select_library_decay`, `extract.import_library_decays`, `extract.select_filter` | Implemented |
| Native extraction | `extract.apply_current`, `extract.apply_all` | Implemented; batch uses the GUI progress callback |
| Phot2conc input and display | `phot2conc.load_directory`, `phot2conc.load_roi_directory`, `phot2conc.select_file`, `phot2conc.set_display`, `phot2conc.set_decay_range` | Implemented |
| Calibration and error mode | `phot2conc.load_calibration`, `phot2conc.set_calibration`, `phot2conc.set_error_mode` | Implemented through the existing calibration-file and input callbacks |
| File and automatic ROI | `phot2conc.set_roi_source`, `phot2conc.set_auto_roi`, `phot2conc.select_roi` | Implemented; component choice follows existing GUI controls |
| Native quantitative calculation | `phot2conc.calculate`, `phot2conc.add_result`, `phot2conc.calculate_all` | Implemented; batch uses the GUI progress callback |
| Result and image export | `phot2conc.set_export_options`, `phot2conc.export_results` | Implemented |
| Frame-resolved values | version-2 PKL plus normal Phot2conc export | Implemented; export-only, not displayed in the GUI |
| Workflow resume/report | external shell workflow using public commands | Implemented outside the TCP command registry |
| General job status/cancellation and filter-library writing | no public command | Not exposed |


The live `system.capabilities` response is the authoritative command inventory. Background-range control, decay-library selection/import and foreground-filter selection are fully represented by public commands. Batch execution calls the native GUI methods so file-specific data are refreshed and progress remains visible. Frame-wise products are stored in PKL and appended during table export rather than being represented as independent TCP commands.

## Public command families

| Family | Purpose |
|---|---|
| `system.*` | Health, version and live command capabilities. |
| `session.inspect` | High-level application and mounted-mode state. |
| `gui.select_mode` | Select the scientific mode through the existing startup/menu callback. |
| `extract.*` | Load PTU data, configure gates and filters, use the decay library, select accepted filters and run extraction. |
| `phot2conc.*` | Load PKL/ROI data, configure displays, calibration, errors and ROI, calculate results and export. |

Arbitrary widget access is deliberately not public. The exact commands and descriptions are returned by `system.capabilities`.

## Protocol contract

Protocol version 1 uses one UTF-8 JSON object per line. Every request carries `protocol_version`, a client-supplied `request_id`, a registered `command` and an object-valued `parameters` member. Authentication is optional and is not returned in logs or responses. Responses echo the request ID and contain either a result or a stable error object.

The normal launcher accepts `--automation-server`, `--automation-port` and timeout/queue options. The server can also be started and stopped from the **About > Automation server** panel. It binds to `127.0.0.1`, applies bounded request and queue limits and never executes Dear PyGui calls on the socket thread.

Machine-readable request and response schemas are stored in `src/smICA/res/schemas/automation_request_v1.schema.json` and `automation_response_v1.schema.json`. Command validators reject unknown fields and normalize accepted parameters before dispatch.

## Workflow contract

The article workflow is an external shell orchestrator that sends validated public commands to one running smICA instance. Dataset order is deterministic. Calibration values for dextran measurements are selected using both the acquisition-date directory and excitation-power directory; the selected JSON supplies the numerical calibration fields sent to Phot2conc.

Automatic ROI selection uses foreground intensity. File-based workflows enumerate ROI names associated with each measurement. Output reuse is controlled by explicit resume state and by checking that the expected completed output is present. Scientific commands continue to use the active GUI state and native callbacks.



Extraction configures gates and statistical filters once for a compatible directory and then calls **Apply to all**. Phot2conc loads calibration and ROI state and calls **Calculate all**, including every matching ROI. The workflow can restrict execution to selected dataset families, for example mRNA measurements only. Completed extraction/calculation batches are recorded in a resume-state file and skipped on restart. At termination, the report records success or failure, input and processed counts, reused/skipped counts, elapsed time, host, operating system, kernel, CPU, logical CPU count, memory and Python version.

## Result report contract

The workflow report records the smICA workflow status, start and finish timestamps, elapsed time, input PTU count, successful extraction/calculation batches, skipped or reused batches, exported result-file and row counts, the resume-state location and machine specification. Result tables retain measurement, ROI and channel identifiers together with integrated and available frame-resolved numerical fields. Authentication data are never included.

## Regression baseline policy

The migration fixture set covers Standard and PIE acquisitions, 128×128 and 256×256 rasters, single- and dual-detector data, and short and long frame series. Validation uses:

1. Exact invariants: metadata, integer photon/marker/frame counts, shapes, selected frame sums, masks when deterministic and file ordering.
2. Floating-point invariants: effective volume, molecules, concentration, uncertainty terms, decay maps and statistical-filter arrays with documented absolute and relative tolerances.
3. Structural invariants: output naming, report status, warnings, per-file failures and resume behaviour.

The historical reader preserves the known earlier effective-frame interpretation only for compatibility with older PKL files. Corrected PicoHarp validation relies on decoded routed-photon totals, marker-derived frame boundaries, agreement between frame sums and the integrated image, analytical numerical examples and manual GUI parity. Current real fixtures do not cover malformed marker streams, missing frames, T2 records, every PicoQuant record type or every scanner; unsupported combinations remain on the explicit legacy route or fail validation.



Reproducibility records use input/output names, sizes, effective parameters, software versions, processing status, warnings, failures, resume state and quantitative summaries. The workflow does not generate or verify checksum files. Scientific agreement is assessed from explicit numerical invariants and tolerances rather than file digests.

## Unautomated or deferred GUI operations

Pure layout controls, display contrast/brightness, ROI-overlay opacity, colour-map display ranges, help windows and file-dialog navigation are intentionally outside the scientific API. Standalone Rewrite ROI and ROI Mixer operations are documented as preparatory tools and are not part of the automated PTU-to-concentration path. Writing new shared filter-library entries is also not exposed because it changes persistent application resources and requires an explicit collision policy. Unsupported reader/backend combinations fail validation instead of silently switching frame semantics.

## Current verification status

- [x] The entry point, scientific modes, common state, packaging and user notebooks have been audited.
- [x] The documented PTU extraction, filtering, calibration, ROI, calculation, export and native batch paths have public commands.
- [x] Commands execute on the GUI thread and reuse existing callbacks.
- [x] Version-2 PKL output contains integrated and frame-resolved products.
- [x] Installed unit tests report expected and calculated values.
- [x] Article workflows support resume state and final processing reports.
- [ ] New PTU hardware/scanner combinations still require representative fixtures and independent numerical validation before support is claimed.